In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count
from schema import transaction_schema
from pyspark.sql import Window, functions as F                                                                                   
from config import processed_data_path, checkpoint_path
import sys                                                                                                                                     
sys.path.append("../..")
from data_processing import process_and_write_batch

In [2]:
spark = SparkSession.builder \
        .appName("KafkaIncrementalBatch") \
        .master("local[*]") \
        .config(
            "spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
        ) \
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.2.0


In [ ]:
df_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 10000)
    .load()
)

parsed_stream_df = (
    df_stream
    .select(
        from_json(
            col("value").cast("string"), 
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

query = (
    parsed_stream_df.writeStream
    .foreachBatch(process_and_write_batch)
    .option("checkpointLocation", checkpoint_path.as_posix())
    # .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()

Processed and wrote batch 0
Processed and wrote batch 1
Processed and wrote batch 2
Processed and wrote batch 3
Processed and wrote batch 4
Processed and wrote batch 5
Processed and wrote batch 6
Processed and wrote batch 7
Processed and wrote batch 8
Processed and wrote batch 9
Processed and wrote batch 10
Processed and wrote batch 11
Processed and wrote batch 12
Processed and wrote batch 13
Processed and wrote batch 14
Processed and wrote batch 15
Processed and wrote batch 16
Processed and wrote batch 17
Processed and wrote batch 18
Processed and wrote batch 19
Processed and wrote batch 20
Processed and wrote batch 21
Processed and wrote batch 22
Processed and wrote batch 23
Processed and wrote batch 24
Processed and wrote batch 25
Processed and wrote batch 26
Processed and wrote batch 27
Processed and wrote batch 28
Processed and wrote batch 29
Processed and wrote batch 30
Processed and wrote batch 31
